# Fine-Tuning do Mistral 7B para Assistente Médico

**Tech Challenge Fase 3 - FIAP Pós-Graduação em IA para Desenvolvedores**

Este notebook realiza o fine-tuning do modelo Mistral 7B Instruct v0.3 usando:
- **QLoRA 4-bit**: Quantização para caber na GPU T4 (15GB)
- **Unsloth**: Biblioteca otimizada para fine-tuning 2x mais rápido
- **Trainer HuggingFace**: Treinamento estável e compatível
- **Dataset**: MedQuAD híbrido (~20k pares QA: 80% PT + 20% EN + BR)

## Configuração para Google Colab Free

- **GPU**: T4 (15GB VRAM)
- **Sessão**: ~12h máximo
- **Estratégia**: Checkpoints a cada 500 steps para poder resumir

## Fluxo do Notebook

1. Configuração do ambiente
2. Montar Google Drive
3. Carregar dataset traduzido
4. Carregar modelo base (Mistral 7B)
5. Configurar LoRA
6. Treinar (com checkpoints)
7. Salvar modelo
8. Testar inferência

---
## 1. Configuração do Ambiente

Instalação das dependências necessárias. Execute esta célula apenas uma vez por sessão.

**Dependências:**
- **Unsloth**: Otimizações para fine-tuning (2x mais rápido)
- **PEFT**: Parameter-Efficient Fine-Tuning (LoRA)
- **Transformers**: Trainer e modelos (já vem com Unsloth)
- **Datasets**: Carregamento de dados

In [ ]:
# ============================================================================
# INSTALAÇÃO DAS DEPENDÊNCIAS
# ============================================================================
# Unsloth: biblioteca otimizada para fine-tuning (2x mais rápido)
# Transformers: Trainer e modelo
# PEFT: Parameter-Efficient Fine-Tuning (LoRA)
# ============================================================================

!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps peft accelerate bitsandbytes -q
!pip install datasets -q

print("✓ Dependências instaladas com sucesso!")

In [ ]:
# ============================================================================
# VERIFICAR GPU DISPONÍVEL
# ============================================================================
# Este notebook requer GPU. No Colab Free, geralmente é uma T4 (15GB).
# Se aparecer "No GPU", vá em Runtime > Change runtime type > GPU
# ============================================================================

import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✓ GPU disponível: {gpu_name}")
    print(f"  Memória total: {gpu_memory:.1f} GB")
else:
    print("✗ ERRO: Nenhuma GPU detectada!")
    print("  Vá em Runtime > Change runtime type > GPU")

---
## 2. Montar Google Drive

O Google Drive será usado para:
- Carregar o dataset traduzido (`training_data.jsonl`)
- Salvar checkpoints durante o treino
- Salvar o modelo final

In [ ]:
# ============================================================================
# MONTAR GOOGLE DRIVE
# ============================================================================
# Isso abrirá uma janela pedindo permissão para acessar seu Drive.
# Após autorizar, seus arquivos estarão em /content/drive/MyDrive/
# ============================================================================

from google.colab import drive
drive.mount('/content/drive')

print("\n✓ Google Drive montado em /content/drive/MyDrive/")

In [ ]:
# ============================================================================
# CONFIGURAÇÃO DE PATHS
# ============================================================================
# IMPORTANTE: Ajuste estes paths de acordo com onde você salvou os arquivos
# no seu Google Drive.
# ============================================================================

import os

# Diretório base no Google Drive
DRIVE_BASE = "/content/drive/MyDrive/FIAP/TechChallenge"

# Paths dos dados
DATA_DIR = f"{DRIVE_BASE}/data"
TRAINING_DATA_PATH = f"{DATA_DIR}/training_data.jsonl"

# Paths dos modelos e checkpoints
MODELS_DIR = f"{DRIVE_BASE}/models"
CHECKPOINT_DIR = f"{MODELS_DIR}/checkpoints"
FINAL_MODEL_DIR = f"{MODELS_DIR}/medical-assistant-final"

# Criar diretórios se não existirem
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

print("Configuração de paths:")
print(f"  Dataset: {TRAINING_DATA_PATH}")
print(f"  Checkpoints: {CHECKPOINT_DIR}")
print(f"  Modelo final: {FINAL_MODEL_DIR}")

In [ ]:
# ============================================================================
# VERIFICAR SE O DATASET EXISTE
# ============================================================================
# Se o arquivo não existir, você precisa fazer upload do training_data.jsonl
# gerado pelo script 03_preparar_dataset.py
# ============================================================================

if os.path.exists(TRAINING_DATA_PATH):
    file_size = os.path.getsize(TRAINING_DATA_PATH) / (1024 * 1024)
    print(f"✓ Dataset encontrado: {TRAINING_DATA_PATH}")
    print(f"  Tamanho: {file_size:.2f} MB")
else:
    print(f"✗ ERRO: Dataset não encontrado!")
    print(f"  Esperado em: {TRAINING_DATA_PATH}")
    print("")
    print("  Para resolver:")
    print("  1. Execute scripts/03_preparar_dataset.py na sua máquina local")
    print("  2. Faça upload de data/processed/training_data.jsonl para:")
    print(f"     {TRAINING_DATA_PATH}")

---
## 3. Carregar e Preparar Dataset

O dataset está no formato Alpaca JSONL:
```json
{"instruction": "...", "input": "pergunta", "output": "resposta"}
```

Vamos:
1. Carregar o JSONL
2. Dividir em treino (90%) e validação (10%)
3. Formatar no template Alpaca para o modelo

In [ ]:
# ============================================================================
# CARREGAR DATASET
# ============================================================================

from datasets import load_dataset

# Carregar do arquivo JSONL
dataset = load_dataset('json', data_files=TRAINING_DATA_PATH, split='train')

print(f"✓ Dataset carregado: {len(dataset)} registros")
print(f"  Colunas: {dataset.column_names}")
print("")
print("Exemplo de registro:")
print(f"  instruction: {dataset[0]['instruction'][:80]}...")
print(f"  input: {dataset[0]['input'][:80]}...")
print(f"  output: {dataset[0]['output'][:80]}...")

In [ ]:
# ============================================================================
# DIVIDIR EM TREINO E VALIDAÇÃO
# ============================================================================
# 90% para treino, 10% para validação
# A validação é usada para monitorar overfitting durante o treino
# ============================================================================

dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset_split['train']
eval_dataset = dataset_split['test']

print(f"✓ Dataset dividido:")
print(f"  Treino: {len(train_dataset)} registros (90%)")
print(f"  Validação: {len(eval_dataset)} registros (10%)")

In [ ]:
# ============================================================================
# ESTATÍSTICAS DO DATASET
# ============================================================================
# Entender o tamanho das perguntas e respostas ajuda a definir max_seq_length
# ============================================================================

import numpy as np

# Calcular comprimentos
input_lengths = [len(x['input']) for x in dataset]
output_lengths = [len(x['output']) for x in dataset]
total_lengths = [len(x['input']) + len(x['output']) for x in dataset]

print("Estatísticas de comprimento (caracteres):")
print("")
print("  Perguntas (input):")
print(f"    Média: {np.mean(input_lengths):.0f}")
print(f"    Máximo: {np.max(input_lengths)}")
print(f"    Percentil 95: {np.percentile(input_lengths, 95):.0f}")
print("")
print("  Respostas (output):")
print(f"    Média: {np.mean(output_lengths):.0f}")
print(f"    Máximo: {np.max(output_lengths)}")
print(f"    Percentil 95: {np.percentile(output_lengths, 95):.0f}")
print("")
print("  Total (input + output):")
print(f"    Média: {np.mean(total_lengths):.0f}")
print(f"    Máximo: {np.max(total_lengths)}")
print(f"    Percentil 95: {np.percentile(total_lengths, 95):.0f}")

---
## 4. Carregar Modelo Base (Mistral 7B)

Usamos o **Unsloth** para carregar o modelo com:
- Quantização 4-bit (reduz de ~14GB para ~4GB de VRAM)
- Otimizações de velocidade (2x mais rápido)

O modelo base é: `mistralai/Mistral-7B-Instruct-v0.3`

In [ ]:
# ============================================================================
# CONFIGURAÇÕES DO MODELO
# ============================================================================

# Modelo base
MODEL_NAME = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"

# Tamanho máximo de sequência (em tokens)
# Baseado nas estatísticas do dataset, 2048 deve cobrir 99%+ dos casos
MAX_SEQ_LENGTH = 2048

# Configuração de precisão
DTYPE = None  # Auto-detectar (float16 para T4)
LOAD_IN_4BIT = True  # Quantização 4-bit para caber na T4

print("Configuração do modelo:")
print(f"  Modelo: {MODEL_NAME}")
print(f"  Max seq length: {MAX_SEQ_LENGTH}")
print(f"  Quantização: 4-bit")

In [ ]:
# ============================================================================
# CARREGAR MODELO E TOKENIZER
# ============================================================================
# Isso pode demorar alguns minutos na primeira vez (download ~4GB)
# ============================================================================

from unsloth import FastLanguageModel

print("Carregando modelo (pode demorar alguns minutos)...")
print("")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

print("")
print("✓ Modelo carregado com sucesso!")

---
## 5. Configurar LoRA

**LoRA (Low-Rank Adaptation)** permite treinar apenas uma pequena fração dos parâmetros:
- Modelo completo: ~7 bilhões de parâmetros
- Com LoRA: ~42 milhões de parâmetros treináveis (~0.6%)

Isso reduz drasticamente o uso de memória e tempo de treino.

In [ ]:
# ============================================================================
# CONFIGURAÇÃO DO LORA
# ============================================================================
# r (rank): Quanto maior, mais capacidade de aprendizado (mas mais memória)
# lora_alpha: Geralmente 2x o valor de r
# target_modules: Quais camadas do modelo serão treinadas
# ============================================================================

model = FastLanguageModel.get_peft_model(
    model,
    r=64,                    # Rank do LoRA (maior = mais capacidade)
    lora_alpha=128,          # Scaling factor (geralmente 2x r)
    lora_dropout=0.05,       # Dropout para regularização
    bias="none",             # Não treinar bias

    # Camadas a serem treinadas:
    # - Attention: q_proj, k_proj, v_proj, o_proj
    # - MLP: gate_proj, up_proj, down_proj
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention
        "gate_proj", "up_proj", "down_proj",      # MLP (importante para domínio médico)
    ],

    # Otimizações do Unsloth
    use_gradient_checkpointing="unsloth",  # Economiza memória
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

# Mostrar estatísticas
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
percent = 100 * trainable_params / total_params

print(f"\n✓ LoRA configurado!")
print(f"  Parâmetros treináveis: {trainable_params:,} ({percent:.2f}%)")
print(f"  Parâmetros totais: {total_params:,}")

---
## 6. Preparar Formato de Treinamento

O modelo espera um formato específico de prompt. Usamos o template **Alpaca** em português:

In [ ]:
# ============================================================================
# TEMPLATE ALPACA EM PORTUGUÊS
# ============================================================================
# Este é o formato que o modelo aprenderá a seguir.
# Durante inferência, usaremos o mesmo formato (sem a resposta).
# ============================================================================

ALPACA_PROMPT = """Abaixo está uma instrução que descreve uma tarefa, junto com uma entrada que fornece contexto adicional. Escreva uma resposta que complete adequadamente a solicitação.

### Instrução:
{}

### Entrada:
{}

### Resposta:
{}"""

# Token de fim de sequência
EOS_TOKEN = tokenizer.eos_token

print("Template Alpaca configurado.")
print("")
print("Exemplo de prompt formatado:")
print("-" * 50)
example = ALPACA_PROMPT.format(
    "Responda a seguinte pergunta médica.",
    "O que é diabetes?",
    "Diabetes é uma doença..."
)
print(example[:300] + "...")

In [ ]:
# ============================================================================
# FUNÇÃO DE FORMATAÇÃO DO DATASET
# ============================================================================
# Converte cada registro do dataset para o formato Alpaca
# ============================================================================

def formatting_prompts_func(examples):
    """
    Formata os exemplos do dataset no template Alpaca.

    Args:
        examples: Batch de exemplos do dataset

    Returns:
        Dict com campo 'text' contendo os prompts formatados
    """
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]

    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        # Formatar no template Alpaca + token de fim
        text = ALPACA_PROMPT.format(instruction, input_text, output) + EOS_TOKEN
        texts.append(text)

    return {"text": texts}

# Aplicar formatação ao dataset
train_dataset_formatted = train_dataset.map(
    formatting_prompts_func,
    batched=True,
    desc="Formatando treino"
)

eval_dataset_formatted = eval_dataset.map(
    formatting_prompts_func,
    batched=True,
    desc="Formatando validação"
)

print(f"\n✓ Dataset formatado!")
print(f"  Treino: {len(train_dataset_formatted)} exemplos")
print(f"  Validação: {len(eval_dataset_formatted)} exemplos")

---
## 7. Configurar Treinamento

Configuramos o `Trainer` do HuggingFace com:
- **5 epochs** para qualidade máxima
- **Checkpoints a cada 500 steps** (crítico para Colab Free!)
- **Avaliação periódica** para monitorar overfitting

Nota: Usamos o Trainer básico do HuggingFace em vez do SFTTrainer do TRL
por ser mais estável e ter menos problemas de compatibilidade de versões.

In [ ]:
# ============================================================================
# CONFIGURAÇÃO DO TREINAMENTO
# ============================================================================
# IMPORTANTE: Checkpoints frequentes para Colab Free!
# Se a sessão desconectar, você pode resumir do último checkpoint.
# ============================================================================

from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

# Argumentos de treinamento
training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,

    # ===== Epochs e Batching =====
    num_train_epochs=5,                    # 5 epochs para qualidade máxima
    per_device_train_batch_size=2,         # Batch size (T4 suporta 2)
    per_device_eval_batch_size=2,          # Batch size para avaliação
    gradient_accumulation_steps=8,         # Effective batch = 2 * 8 = 16

    # ===== Learning Rate =====
    learning_rate=2e-4,                    # Taxa de aprendizado
    warmup_steps=200,                      # ~3% warmup (total ~5800 steps)
    weight_decay=0.01,                     # Regularização
    lr_scheduler_type="cosine",            # Decay suave

    # ===== Precisão =====
    fp16=True,                             # Float16 para T4

    # ===== Checkpoints (CRÍTICO para Colab Free!) =====
    save_strategy="steps",
    save_steps=500,                        # Salvar a cada 500 steps (~30-40 min)
    save_total_limit=3,                    # Manter apenas 3 checkpoints (economiza espaço)

    # ===== Avaliação =====
    eval_strategy="steps",                 # Estratégia de avaliação
    eval_steps=500,                        # Avaliar junto com checkpoint
    load_best_model_at_end=True,           # Carregar melhor modelo no final
    metric_for_best_model="eval_loss",     # Métrica para escolher melhor

    # ===== Logging =====
    logging_steps=50,                      # Log a cada 50 steps
    report_to="none",                      # Desabilitar W&B

    # ===== Otimizações =====
    optim="adamw_8bit",                    # Otimizador 8-bit (economiza memória)
    seed=42,
)

print("✓ TrainingArguments configurado!")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size efetivo: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Checkpoints: a cada {training_args.save_steps} steps")

In [ ]:
# ============================================================================
# CRIAR TRAINER
# ============================================================================
# Usamos o Trainer básico do HuggingFace (mais estável que SFTTrainer)
# ============================================================================

# Tokenizar o dataset COM labels
def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
    )
    # Para language modeling, labels = input_ids
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

print("Tokenizando datasets...")
train_tokenized = train_dataset_formatted.map(
    tokenize_function, 
    batched=True, 
    remove_columns=["text", "instruction", "input", "output"],
    desc="Tokenizando treino"
)
eval_tokenized = eval_dataset_formatted.map(
    tokenize_function, 
    batched=True, 
    remove_columns=["text", "instruction", "input", "output"],
    desc="Tokenizando validação"
)

# Data collator para language modeling (causal LM)
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, 
    mlm=False,  # Causal LM, não masked
)

# Criar Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    data_collator=data_collator,
)

print("✓ Trainer criado!")

---
## 8. Verificar Checkpoints Existentes

**IMPORTANTE para Colab Free!**

Se a sessão desconectou, esta célula encontra o último checkpoint para continuar o treino.

In [ ]:
# ============================================================================
# ENCONTRAR ÚLTIMO CHECKPOINT
# ============================================================================
# Se existir um checkpoint, podemos continuar de onde paramos.
# Isso é ESSENCIAL para Colab Free (sessões de ~12h)
# ============================================================================

def find_latest_checkpoint(checkpoint_dir):
    """
    Encontra o checkpoint mais recente no diretório.

    Returns:
        Path do checkpoint ou None se não existir
    """
    if not os.path.exists(checkpoint_dir):
        return None

    checkpoints = [
        d for d in os.listdir(checkpoint_dir)
        if d.startswith("checkpoint-") and os.path.isdir(os.path.join(checkpoint_dir, d))
    ]

    if not checkpoints:
        return None

    # Ordenar por número do step
    checkpoints_sorted = sorted(
        checkpoints,
        key=lambda x: int(x.split("-")[1])
    )

    latest = checkpoints_sorted[-1]
    return os.path.join(checkpoint_dir, latest)

# Verificar
latest_checkpoint = find_latest_checkpoint(CHECKPOINT_DIR)

if latest_checkpoint:
    print(f"🔄 Checkpoint encontrado!")
    print(f"   {latest_checkpoint}")
    print("")
    print("   O treino será RETOMADO deste ponto.")
    print("   Se quiser começar do zero, delete a pasta de checkpoints.")
else:
    print("🆕 Nenhum checkpoint encontrado.")
    print("   O treino começará do zero.")

---
## 9. Treinar!

Esta é a célula principal de treinamento.

**Tempo estimado (Colab Free - T4):**
- 1 epoch: ~4-6 horas
- 5 epochs: ~20-30 horas (3-4 sessões)

**Se a sessão desconectar:**
1. Reconecte ao Colab
2. Execute as células 1-8 novamente
3. Execute esta célula - ela retomará do último checkpoint automaticamente

In [ ]:
# ============================================================================
# TREINAR O MODELO
# ============================================================================
# Se existir checkpoint, retoma automaticamente.
# Checkpoints são salvos a cada 500 steps no Google Drive.
# ============================================================================

print("=" * 60)
print("  INICIANDO TREINAMENTO")
print("=" * 60)
print("")

if latest_checkpoint:
    print(f"🔄 Retomando de: {latest_checkpoint}")
    print("")
    trainer_stats = trainer.train(resume_from_checkpoint=latest_checkpoint)
else:
    print("🆕 Iniciando treino do zero...")
    print("")
    trainer_stats = trainer.train()

print("")
print("=" * 60)
print("  TREINAMENTO CONCLUÍDO!")
print("=" * 60)

In [ ]:
# ============================================================================
# ESTATÍSTICAS DO TREINAMENTO
# ============================================================================

print("Estatísticas do treinamento:")
print(f"  Tempo total: {trainer_stats.metrics['train_runtime'] / 3600:.2f} horas")
print(f"  Steps totais: {trainer_stats.metrics['train_steps']}")
print(f"  Loss final: {trainer_stats.metrics['train_loss']:.4f}")
print(f"  Samples/segundo: {trainer_stats.metrics['train_samples_per_second']:.2f}")

---
## 10. Salvar Modelo Final

Salvamos os **adapters LoRA** (não o modelo completo):
- Tamanho: ~500MB (vs ~15GB do modelo completo)
- Para usar: carregar modelo base + aplicar adapters

In [ ]:
# ============================================================================
# SALVAR MODELO FINAL
# ============================================================================
# Salvamos os adapters LoRA e o tokenizer.
# Para usar depois: carregar modelo base + aplicar estes adapters.
# ============================================================================

print(f"Salvando modelo em: {FINAL_MODEL_DIR}")
print("")

# Salvar adapters LoRA
model.save_pretrained(FINAL_MODEL_DIR)
print("✓ Adapters LoRA salvos")

# Salvar tokenizer
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print("✓ Tokenizer salvo")

# Listar arquivos salvos
print("")
print("Arquivos salvos:")
for f in os.listdir(FINAL_MODEL_DIR):
    size = os.path.getsize(os.path.join(FINAL_MODEL_DIR, f)) / (1024 * 1024)
    print(f"  {f}: {size:.2f} MB")

total_size = sum(
    os.path.getsize(os.path.join(FINAL_MODEL_DIR, f))
    for f in os.listdir(FINAL_MODEL_DIR)
) / (1024 * 1024)
print(f"")
print(f"Tamanho total: {total_size:.2f} MB")

---
## 11. Testar Inferência

Vamos testar o modelo fine-tuned com algumas perguntas médicas em português.

In [ ]:
# ============================================================================
# PREPARAR MODELO PARA INFERÊNCIA
# ============================================================================

FastLanguageModel.for_inference(model)
print("✓ Modelo preparado para inferência")

In [ ]:
# ============================================================================
# FUNÇÃO DE INFERÊNCIA
# ============================================================================

def ask_medical_question(question: str, max_new_tokens: int = 512) -> str:
    """
    Faz uma pergunta médica ao modelo fine-tuned.

    Args:
        question: Pergunta em português
        max_new_tokens: Máximo de tokens na resposta

    Returns:
        Resposta do modelo
    """
    # Formatar prompt
    prompt = ALPACA_PROMPT.format(
        "Responda a seguinte pergunta médica de forma clara e detalhada.",
        question,
        ""  # Resposta vazia - modelo vai completar
    )

    # Tokenizar
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    # Gerar resposta
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        temperature=0.7,
        top_p=0.9,
    )

    # Decodificar
    response = tokenizer.batch_decode(outputs)[0]

    # Extrair apenas a resposta (após "### Resposta:")
    if "### Resposta:" in response:
        response = response.split("### Resposta:")[1].strip()
        # Remover token de fim se presente
        if EOS_TOKEN in response:
            response = response.split(EOS_TOKEN)[0].strip()

    return response

In [ ]:
# ============================================================================
# TESTAR COM PERGUNTAS MÉDICAS
# ============================================================================

perguntas_teste = [
    "Quais são os sintomas do diabetes tipo 2?",
    "Como é feito o diagnóstico de hipertensão arterial?",
    "Quais são os tratamentos disponíveis para asma?",
]

print("=" * 60)
print("  TESTE DE INFERÊNCIA")
print("=" * 60)

for i, pergunta in enumerate(perguntas_teste, 1):
    print(f"\n--- Pergunta {i} ---")
    print(f"Q: {pergunta}")
    print("")

    resposta = ask_medical_question(pergunta)

    print(f"R: {resposta}")
    print("-" * 40)

---
## 12. Carregar Modelo Salvo (Para Uso Futuro)

Esta célula mostra como carregar o modelo fine-tuned em uma sessão futura.

In [ ]:
# ============================================================================
# COMO CARREGAR O MODELO EM UMA SESSÃO FUTURA
# ============================================================================
# Execute este código para carregar o modelo fine-tuned depois.
# Não precisa re-treinar!
# ============================================================================

# Descomente para usar:

# from unsloth import FastLanguageModel
#
# # Path do modelo salvo
# MODEL_PATH = "/content/drive/MyDrive/FIAP/TechChallenge/models/medical-assistant-final"
#
# # Carregar modelo fine-tuned
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=MODEL_PATH,
#     max_seq_length=2048,
#     dtype=None,
#     load_in_4bit=True,
# )
#
# # Preparar para inferência
# FastLanguageModel.for_inference(model)
#
# print("✓ Modelo carregado e pronto para uso!")

print("Código para carregar modelo em sessão futura (descomente para usar)")

---
## Próximos Passos

Após o fine-tuning:

1. **Avaliação Baseline vs Fine-Tuned**
   - Execute `scripts/05_avaliar_baseline.py` (Mistral base)
   - Execute `scripts/06_avaliar_finetuned.py` (modelo treinado)
   - Compare as respostas lado a lado

2. **Integração com LangGraph**
   - O modelo fine-tuned será usado pelo agente `raciocinio` no grafo multi-agente

3. **Documentação**
   - Adicionar curvas de loss ao relatório LaTeX
   - Documentar hiperparâmetros usados
   - Incluir exemplos de antes/depois

In [ ]:
# ============================================================================
# FIM DO NOTEBOOK
# ============================================================================

print("🎉 Fine-tuning concluído com sucesso!")
print("")
print(f"Modelo salvo em: {FINAL_MODEL_DIR}")
print("")
print("Próximos passos:")
print("  1. Executar scripts de avaliação (05 e 06)")
print("  2. Integrar com LangGraph")
print("  3. Documentar resultados no relatório")